# Step 03. 속도 계획 (Velocity Planning)

> **출처**: PythonRobotics (https://github.com/AtsushiSakai/PythonRobotics)  
> Original author: Atsushi Sakai (@Atsushi_twi) / License: MIT  
>  
> **교육 주관**: Hanyang University, Automotive Intelligence Laboratory (AILAB)

Step 02에서 Frenet 좌표계로 Reference 경로를 표현했습니다.  
경로의 **형태(x, y)** 는 정해졌지만 각 지점에서 **얼마나 빠르게** 달릴지는 아직 미정입니다.  
이번 단계에서는 곡률과 가·감속 한계를 반영해 **속도 프로파일**을 단계별로 만들어 갑니다.

## 학습 목표

1. 곡률 = 1/회전반경 의 의미를 **접촉원 애니메이션**으로 직관적으로 이해
2. 속도 계획 4단계를 **각 단계의 입력→출력** 그래프로 분리하여 이해
   - ① 기준 속도 초기화 → ② 곡률 제한 → ③ Forward → ④ Backward
3. Forward(감속 한계) / Backward(가속 한계) 패스가 프로파일을 어떻게 바꾸는지 확인
4. 최종 속도 프로파일로 주행하는 차량 시각화

## 0. 예시 주행 궤적

### 이론 배경

속도 계획의 입력은 **경로의 곡률 프로파일**입니다.  
아래 궤적을 곡률로 색칠하면 어디가 급커브(붉은색)인지 보입니다.  
급커브에서는 원심력 때문에 속도를 낮춰야 합니다.

In [ ]:
%matplotlib inline
import sys, os, math
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
from IPython.display import HTML

from utils.cubic_spline import calc_spline_course

# 예시 주행 궤적 (Day 2에서 계속 사용하는 경로)
wx = [0.0, 10.0, 20.5, 35.0, 50.0, 60.0]
wy = [0.0, -6.0,  5.0,  6.5,  0.0, -3.0]
rx, ry, ryaw, rk, rs = calc_spline_course(wx, wy, ds=0.5)

print('예시 주행 궤적 생성 완료')
print(f'  경로 길이      : {rs[-1]:.1f} m')
print(f'  경로 점 개수   : {len(rx)}개  (ds=0.5m)')
print(f'  최대 곡률      : {max(abs(k) for k in rk):.4f} rad/m')
print(f'  최소 회전반경  : {1/max(abs(k) for k in rk):.1f} m  (가장 급한 커브)')

# 곡률로 색칠한 궤적 — 어디가 급커브인지 한눈에
fig, ax = plt.subplots(figsize=(11, 5))
sc = ax.scatter(rx, ry, c=[abs(k) for k in rk], cmap='autumn_r', s=15)
ax.plot(wx, wy, 'o', ms=9, color='navy', alpha=0.6, label='Waypoints')
ax.plot(rx[0], ry[0], 'og', ms=12, label='Start')
ax.plot(rx[-1], ry[-1], 'sb', ms=11, label='End')
plt.colorbar(sc, ax=ax, label='|curvature| [rad/m]', fraction=0.04)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=8)
ax.set_title('Example Trajectory (red = sharp curve = must slow down)')
plt.tight_layout()
plt.show()
print('붉은 구간 = 곡률 큰 급커브 -> 속도를 낮춰야 하는 곳')


## 1. 곡률과 회전반경 — 접촉원(Osculating Circle)

### 이론 배경

곡률 $\kappa$ 와 회전반경 $R$ 은 역수 관계입니다.

$$R = \frac{1}{|\kappa|}$$

각 점에서 경로에 **접하는 원**(접촉원)의 반지름이 그 점의 회전반경입니다.

- **직선** 구간: $\kappa \to 0$, 원이 무한히 커짐 (거의 직선)
- **급커브** 구간: $\kappa$ 큼, 원이 작아짐

아래 애니메이션에서 빨간 점이 경로를 따라가며 주황색 접촉원의 크기가 변합니다.  
오른쪽 곡률 그래프의 빨간 선이 현재 위치를 표시합니다.  
**작은 원 = 큰 곡률 = 낮은 허용 속도** 라는 관계가 핵심입니다.

In [ ]:
# 곡률 = 1/회전반경.  각 점에서 경로에 접하는 원(접촉원)의 반지름이 회전반경.
# 직선에서는 원이 매우 커지고(반경 무한대), 급커브에서는 원이 작아진다.
R_DISPLAY_CAP = 45.0   # 시각화용 반경 상한 (직선부 무한대 방지)

def osculating_circle(i):
    k = rk[i]
    if abs(k) < 1e-4:
        return None, float('inf')           # 거의 직선
    R = 1.0 / k                              # 부호 포함 (좌/우 곡선 구분)
    cx = rx[i] - math.sin(ryaw[i]) * R
    cy = ry[i] + math.cos(ryaw[i]) * R
    return (cx, cy), abs(R)

SKIP = 3
frames_idx = list(range(0, len(rx), SKIP))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 6),
                               gridspec_kw={'width_ratios':[1.4, 1]})

# 왼쪽: 경로 + 움직이는 접촉원
axL.plot(rx, ry, '-', color='steelblue', lw=2, alpha=0.6)
axL.set_aspect('equal'); axL.grid(True, alpha=0.3)
axL.set_xlim(min(rx)-12, max(rx)+12); axL.set_ylim(min(ry)-14, max(ry)+14)
axL.set_title('Osculating Circle (radius = 1 / curvature)')
car_dot,   = axL.plot([], [], 'o', color='red', ms=11, zorder=6)
circle_pt  = Circle((0,0), 0, fill=False, color='darkorange', lw=2, alpha=0.9)
axL.add_patch(circle_pt)
center_dot,= axL.plot([], [], '+', color='darkorange', ms=10, mew=2)
radius_ln, = axL.plot([], [], '--', color='darkorange', lw=1)
info = axL.text(0.02, 0.97, '', transform=axL.transAxes, va='top', fontsize=10,
                bbox=dict(boxstyle='round', fc='white', alpha=0.7))

# 오른쪽: 곡률 프로파일 + 현재 위치
axR.plot(rs, [abs(k) for k in rk], '-', color='purple', lw=1.5)
axR.fill_between(rs, [abs(k) for k in rk], alpha=0.2, color='purple')
axR.set_xlabel('Arc length s [m]'); axR.set_ylabel('|curvature| [rad/m]')
axR.set_title('Curvature along Path'); axR.grid(True, alpha=0.3)
vmark = axR.axvline(0, color='red', lw=1.5)

def update_osc(fi):
    i = frames_idx[fi]
    car_dot.set_data([rx[i]], [ry[i]])
    center, R = osculating_circle(i)
    if center is None or R > R_DISPLAY_CAP:
        circle_pt.set_radius(0); center_dot.set_data([], []); radius_ln.set_data([], [])
        info.set_text(f's = {rs[i]:.1f} m\nR = inf (near-straight)\nkappa = {rk[i]:+.3f}')
    else:
        circle_pt.center = center; circle_pt.set_radius(R)
        center_dot.set_data([center[0]], [center[1]])
        radius_ln.set_data([center[0], rx[i]], [center[1], ry[i]])
        info.set_text(f's = {rs[i]:.1f} m\nR = {R:.1f} m\nkappa = {rk[i]:+.3f}')
    vmark.set_xdata([rs[i]])
    return car_dot, circle_pt, center_dot, radius_ln, info, vmark

ani_osc = FuncAnimation(fig, update_osc, frames=len(frames_idx),
                        interval=90, repeat=False, blit=False)
plt.tight_layout()
HTML(ani_osc.to_jshtml())


## 2. 단계 ① — 기준 속도 초기화

### 이론 배경

출발은 단순합니다. 모든 점을 **기준 속도(예: 30 km/h)** 로 평평하게 초기화합니다.  
이후 단계에서 이 속도를 **아래로만** 조정합니다 (안전 원칙: 기준 속도 초과 없음).

- **입력**: 없음 (상수)
- **출력**: 평평한 속도 프로파일

In [ ]:
TARGET_SPEED = 30.0 / 3.6  # m/s (30 km/h)
MAX_ACCEL    = 2.0          # m/s^2
MAX_DECEL    = 3.0          # m/s^2

# 단계 ①: 모든 점을 기준 속도로 초기화 (아직 곡률 무시)
v_stage1 = [TARGET_SPEED] * len(rx)

print('단계 ① 기준 속도 초기화')
print(f'  기준 속도: {TARGET_SPEED*3.6:.0f} km/h 로 전 구간 동일하게 설정')

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(rs, [v*3.6 for v in v_stage1], '-', color='gray', lw=2.5, label='① Base speed (flat)')
ax.fill_between(rs, [v*3.6 for v in v_stage1], alpha=0.15, color='gray')
ax.set_xlabel('Arc length s [m]'); ax.set_ylabel('Speed [km/h]')
ax.set_ylim(0, 35); ax.grid(True, alpha=0.3); ax.legend(loc='lower right')
ax.set_title('Stage 1: Initialize with constant base speed')
plt.tight_layout()
plt.show()
print('  -> 입력: 없음 / 출력: 평평한 속도 프로파일 (곡률 아직 반영 안됨)')


## 3. 단계 ② — 곡률 기반 속도 제한

### 이론 배경

횡방향 가속도 한계 $a_{lat,max}$ 로부터 허용 최대 속도를 유도합니다.

$$a_{lat} = v^2 \kappa \le a_{lat,max} \;\Rightarrow\; v_{max} = \frac{K_{coeff}}{\sqrt{|\kappa|}}, \quad K_{coeff} = \sqrt{a_{lat,max}}$$

곡률이 큰 곳에서 $v_{max}$ 가 낮아집니다.

- **입력**: 기준 속도 + 곡률 프로파일
- **출력**: 커브에서 깎인 속도 (단, 아직 톱니 모양 → 급가감속이라 비현실적)

In [ ]:
K_COEFF = 1.0  # sqrt(a_lat_max) 에 해당

def curvature_speed_limit(curvatures, k_coeff=K_COEFF, min_speed=1.0):
    # v_max = k_coeff / sqrt(|kappa|)  (횡가속도 a_lat = v^2 * kappa 제한에서 유도)
    out = []
    for k in curvatures:
        if abs(k) < 1e-6:
            out.append(TARGET_SPEED)
        else:
            out.append(max(min_speed, k_coeff / math.sqrt(abs(k))))
    return out

v_curv   = curvature_speed_limit(rk)
v_stage2 = [min(v_stage1[i], v_curv[i]) for i in range(len(rx))]

reduced = [i for i in range(len(rx)) if v_stage2[i] < v_stage1[i] - 0.01]
print('단계 ② 곡률 기반 속도 제한')
print(f'  곡률 때문에 감속된 구간: {len(reduced)}개 점')
print(f'  최저 속도: {min(v_stage2)*3.6:.1f} km/h (가장 급한 커브)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                               gridspec_kw={'height_ratios':[1, 2]})
ax1.plot(rs, [abs(k) for k in rk], '-', color='purple', lw=1.5)
ax1.fill_between(rs, [abs(k) for k in rk], alpha=0.2, color='purple')
ax1.set_ylabel('|curvature|'); ax1.set_title('Curvature (input)'); ax1.grid(True, alpha=0.3)

ax2.plot(rs, [v*3.6 for v in v_stage1], '--', color='gray', lw=1.5, label='① Base speed')
ax2.plot(rs, [v*3.6 for v in v_stage2], '-',  color='orange', lw=2.5, label='② Curvature limited')
ax2.fill_between(rs, [v*3.6 for v in v_stage2], alpha=0.15, color='orange')
# 감속된 구간 강조
ax2.plot([rs[i] for i in reduced], [v_stage2[i]*3.6 for i in reduced],
         '.', color='red', ms=4, label='Slowed down')
ax2.set_xlabel('Arc length s [m]'); ax2.set_ylabel('Speed [km/h]')
ax2.set_ylim(0, 35); ax2.grid(True, alpha=0.3); ax2.legend(loc='lower right', fontsize=8)
ax2.set_title('Stage 2: Curvature pushes speed down at sharp curves')
plt.tight_layout()
plt.show()
print('  -> 입력: 기준속도+곡률 / 출력: 커브에서 깎인 속도 (아직 톱니 모양 = 비현실적)')


## 4. 단계 ③ — 전방 스무딩 (Forward Pass)

### 이론 배경

톱니 프로파일은 순간 가·감속이 무한대라 차량이 따라갈 수 없습니다.  
**감속 한계** 내에서 슬로우 존에 **미리** 감속하도록 다듬습니다.

$$v_{i+1} \le \sqrt{v_i^2 + 2 \, a_{decel} \, \Delta s}$$

앞→뒤 방향으로 진행하며, 슬로우 존 진입 전부터 감속이 시작됩니다.

- **입력**: 단계 ② 속도
- **출력**: 감속이 부드럽게 시작되는 프로파일

In [ ]:
def forward_smooth(v_profile, ds=0.5, max_decel=MAX_DECEL):
    # 앞->뒤: v[i+1] <= sqrt(v[i]^2 + 2*max_decel*ds)
    # "지금 속도에서 급정거로 줄일 수 있는 한계"로 다음 속도를 제한 -> 감속 구간이 부드러워짐
    v = list(v_profile)
    for i in range(len(v) - 1):
        v[i+1] = min(v[i+1], math.sqrt(v[i]**2 + 2.0*max_decel*ds))
    return v

v_stage3 = forward_smooth(v_stage2)
changed = [i for i in range(len(rx)) if abs(v_stage3[i]-v_stage2[i]) > 0.01]

print('단계 ③ 전방 스무딩 (Forward Pass)')
print(f'  변화된 점: {len(changed)}개  (감속 한계를 넘던 구간이 완만해짐)')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rs, [v*3.6 for v in v_stage2], '--', color='orange', lw=1.5, alpha=0.8, label='② Curvature limited (before)')
ax.plot(rs, [v*3.6 for v in v_stage3], '-',  color='green',  lw=2.5, label='③ After forward pass')
ax.fill_between(rs, [v*3.6 for v in v_stage3], alpha=0.12, color='green')
ax.plot([rs[i] for i in changed], [v_stage3[i]*3.6 for i in changed],
        '.', color='red', ms=5, label='Changed by forward pass')
ax.set_xlabel('Arc length s [m]'); ax.set_ylabel('Speed [km/h]')
ax.set_ylim(0, 35); ax.grid(True, alpha=0.3); ax.legend(loc='lower right', fontsize=8)
ax.set_title('Stage 3: Forward pass limits how fast we approach a slow zone (decel limit)')
plt.tight_layout()
plt.show()
print('  -> 입력: 단계② 속도 / 출력: 감속이 미리 시작되는 부드러운 프로파일')


## 5. 단계 ④ — 후방 스무딩 (Backward Pass)

### 이론 배경

이번에는 슬로우 존을 빠져나올 때의 **가속 한계**를 반영합니다.

$$v_i \le \sqrt{v_{i+1}^2 + 2 \, a_{accel} \, \Delta s}$$

뒤→앞 방향으로 진행합니다.  
Forward(감속) + Backward(가속) 를 합치면 **가·감속 모두 물리 한계 내** → 실현 가능한 프로파일.

- **입력**: 단계 ③ 속도
- **출력**: 최종 속도 프로파일

In [ ]:
def backward_smooth(v_profile, ds=0.5, max_accel=MAX_ACCEL):
    # 뒤->앞: v[i] <= sqrt(v[i+1]^2 + 2*max_accel*ds)
    # "다음 속도까지 가속 능력으로 도달 가능한 한계"로 현재 속도 제한 -> 가속 구간이 부드러워짐
    v = list(v_profile)
    for i in range(len(v)-2, -1, -1):
        v[i] = min(v[i], math.sqrt(v[i+1]**2 + 2.0*max_accel*ds))
    return v

v_stage4 = backward_smooth(v_stage3)
changed4 = [i for i in range(len(rx)) if abs(v_stage4[i]-v_stage3[i]) > 0.01]

print('단계 ④ 후방 스무딩 (Backward Pass) — 최종')
print(f'  변화된 점: {len(changed4)}개  (가속 한계를 넘던 구간이 완만해짐)')
print(f'  최종 평균 속도: {np.mean(v_stage4)*3.6:.1f} km/h')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(rs, [v*3.6 for v in v_stage3], '--', color='green', lw=1.5, alpha=0.8, label='③ After forward (before)')
ax.plot(rs, [v*3.6 for v in v_stage4], '-',  color='blue',  lw=2.5, label='④ Final (after backward)')
ax.fill_between(rs, [v*3.6 for v in v_stage4], alpha=0.12, color='blue')
ax.plot([rs[i] for i in changed4], [v_stage4[i]*3.6 for i in changed4],
        '.', color='red', ms=5, label='Changed by backward pass')
ax.set_xlabel('Arc length s [m]'); ax.set_ylabel('Speed [km/h]')
ax.set_ylim(0, 35); ax.grid(True, alpha=0.3); ax.legend(loc='lower right', fontsize=8)
ax.set_title('Stage 4: Backward pass limits how fast we accelerate out of a slow zone (accel limit)')
plt.tight_layout()
plt.show()
print('  -> 입력: 단계③ 속도 / 출력: 가속+감속 모두 물리 한계 내 = 실현 가능한 최종 프로파일')


## 6. 종합 및 주행 시각화

### 이론 배경

4단계를 겹쳐 보면 속도가 어떻게 **점진적으로** 다듬어졌는지 보입니다.  
마지막으로 최종 프로파일대로 주행하는 차량을 시각화합니다 — 커브에서 감속(빨강), 직선에서 가속(초록).

다음: Step 04에서 여러 후보 경로를 생성하고 충돌을 검사합니다.

In [ ]:
# 4단계를 한 그래프에 겹쳐 비교
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rs, [v*3.6 for v in v_stage1], '--', color='gray',   lw=1,   alpha=0.6, label='① Base')
ax.plot(rs, [v*3.6 for v in v_stage2], '-',  color='orange', lw=1.3, alpha=0.7, label='② Curvature limit')
ax.plot(rs, [v*3.6 for v in v_stage3], '-',  color='green',  lw=1.3, alpha=0.7, label='③ Forward')
ax.plot(rs, [v*3.6 for v in v_stage4], '-',  color='blue',   lw=2.8,            label='④ Final')
ax.set_xlabel('Arc length s [m]'); ax.set_ylabel('Speed [km/h]')
ax.set_ylim(0, 35); ax.grid(True, alpha=0.3); ax.legend(loc='lower right')
ax.set_title('All 4 Stages Overlaid')
plt.tight_layout()
plt.show()

# 차량이 최종 속도로 주행 — 위치를 속도 색으로 표시 + 속도 그래프에 현재 위치
SKIP = 4
n_frames = math.ceil(len(rx) / SKIP)
v_kmh = np.array(v_stage4) * 3.6

fig, (axP, axV) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={'height_ratios':[2,1]})
sc = axP.scatter(rx, ry, c=v_kmh, cmap='RdYlGn', vmin=v_kmh.min(), vmax=v_kmh.max(), s=10, alpha=0.5)
plt.colorbar(sc, ax=axP, label='Speed [km/h]', fraction=0.02)
dotP, = axP.plot([], [], 'o', color='navy', ms=12, zorder=5)
axP.set_aspect('equal'); axP.grid(True, alpha=0.3)
axP.set_title('Driving the final profile (green=fast, red=slow)')

axV.plot(rs, v_kmh, '-b', lw=2)
axV.fill_between(rs, v_kmh, alpha=0.12, color='blue')
axV.set_xlabel('Arc length s [m]'); axV.set_ylabel('Speed [km/h]'); axV.grid(True, alpha=0.3)
vlineV = axV.axvline(0, color='red', lw=1.5)
txtV = axV.text(0.01, 0.9, '', transform=axV.transAxes, color='red', fontsize=10)

def update_drive(frame):
    k = min(frame*SKIP, len(rx)-1)
    dotP.set_data([rx[k]], [ry[k]])
    vlineV.set_xdata([rs[k]])
    txtV.set_text(f's={rs[k]:.1f}m  v={v_kmh[k]:.1f}km/h')
    return dotP, vlineV, txtV

ani_drive = FuncAnimation(fig, update_drive, frames=n_frames, interval=60, repeat=False, blit=False)
plt.tight_layout()
HTML(ani_drive.to_jshtml())
